1. Fine-tune BERT for sentiment analysis on the IMDb dataset.

2. Fine-tune GPT-2 on the Shakespeare dataset (from Chapter 14), then generate Shakespeare-like text.

3. Download the Wikipedia Movie Plots dataset, and use SBERT to embed every movie description. Then write a function that takes a search query, embeds it, finds the most similar embeddings, and lists the corresponding movies.

4. Use an instruction-tuned model such as Qwen-7B-Instruct to build a little chatbot which acts like a movie expert. Then try adding some RAG functionality, for example by automatically injecting the most relevant movie plot into the prompt (see the previous exercise).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import BertConfig, BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, DataCollator
from huggingface_hub import login

In [2]:
imdb_dataset = load_dataset('imdb')
split = imdb_dataset['train'].train_test_split(train_size=0.8)
imdb_train_set, imdb_valid_set = split['train'], split['test']
imdb_test_set = imdb_dataset['test']

Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at /home/damian/.cache/huggingface/datasets/imdb/plain_text/0.0.0/e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Fri Aug 14 20:04:51 2026).


In [ ]:
#erase later
login('ACCESS_TOKEN')
bert_tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

def collate_fn(batch):
    reviews = [review['text'] for review in batch]
    labels = [[label['label']] for label in batch]
    encodings = bert_tokenizer(reviews, padding=True, truncation=True, max_length=128, return_tensors='pt')
    labels = torch.tensor(labels, dtype=torch.float32)
    return encodings, labels

batch_size = 64
imdb_train_loader = DataLoader(imdb_train_set, batch_size, collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size, collate_fn=collate_fn)
imdb_test_loader  = DataLoader(imdb_test_set,  batch_size, collate_fn=collate_fn)

In [68]:
def tokenize(example, tokenizer=bert_tokenizer):
    return tokenizer(example['text'], truncation=True, max_length=128, padding='max_length')

imdb_train = imdb_train_set.map(tokenize, batched=True)

In [69]:
config = BertConfig(vocab_size=bert_tokenizer.vocab_size, hidden_size=128, num_hidden_layers=2,
                    num_attention_heads=4, intermediate_size=512, max_position_embeddings=128)
bert = BertForSequenceClassification(config)
args = TrainingArguments(output_dir='./my_bert_ex', per_device_train_batch_size=16, num_train_epochs=5)
#collator = DataCollator(tokenizer=bert_tokenizer)
bert_trainer = Trainer(bert, args=args, train_dataset=imdb_train)
trainer_output = bert_trainer.train()

Step,Training Loss
500,0.693015
1000,0.551436
1500,0.426028
2000,0.367064
2500,0.363696
3000,0.307979
3500,0.293948
4000,0.287541
4500,0.261444
5000,0.259731


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 102.29it/s]


In [26]:
imdb_valid_set[0]['label']
imdb_valid_set[1]['text']

'*Spoilers and extreme bashing lay ahead*<br /><br />When this show first started, I found it tolerable and fun. Fairly Oddparents was the kind of cartoon that kids and adults liked. It also had high ratings along with Spongebob. But it started to fall because of the following crap that Butch Hartman and his team shoved into the show.<br /><br />First off, toilet humor isn\'t all that funny. You can easily pull off a fast laugh from a little kiddie with a burp, but that\'s pretty much the only audience that would laugh at such a cliché joke. Next there are the kiddie jokes. Lol we can see people in their underwear and we can see people cross-dressing. LOLOLOL!!! I just can\'t stop laughing at such gay bliss! Somebody help me! But of course, this show wouldn\'t suck that bad if it weren\'t for stereotypes. Did you see how the team portrayed Australians? They saw them as nothing but kangaroo-loving, boomerang-throwing simpletons who live in a hot desert. But now... Is the coup de grace o

In [63]:
imdb_valid = imdb_valid_set.map(tokenize, batched=True)

Map: 100%|██████████| 5000/5000 [00:00<00:00, 7382.35 examples/s]


In [70]:
imdb_valid

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5000
})

In [94]:
bert.eval()
counter = 0
with torch.no_grad():
    for X in imdb_valid:
        y = X['label']
        output = bert(input_ids=torch.tensor(X['input_ids']).unsqueeze(0).to(bert.device),
            token_type_ids=torch.tensor(X['token_type_ids']).unsqueeze(0).to(bert.device),
            attention_mask=torch.tensor(X['attention_mask']).unsqueeze(0).to(bert.device)
        )

        print(y, ' answ: ',output.logits.argmax().item())
        counter += 1
        if counter == 20:
            break

0  answ:  0
0  answ:  1
0  answ:  1
0  answ:  0
0  answ:  0
1  answ:  0
1  answ:  1
0  answ:  0
1  answ:  1
1  answ:  1
0  answ:  0
1  answ:  1
1  answ:  0
0  answ:  0
1  answ:  1
0  answ:  0
0  answ:  0
1  answ:  1
1  answ:  1
1  answ:  1
